<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

In [1]:
# Parameters
beta = 0.0
disease = "BIPOLAR"


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
DISEASE = input("Disease: ")
interlayer_transition_prob = input("Beta: ")
average_t = [2,4,6,8]

StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

# Import

In [ ]:
# ----------------------------
# Standard library
# ----------------------------
import json
import random

# ----------------------------
# Core scientific stack
# ----------------------------
import numpy as np
import scipy.sparse as sp

# ----------------------------
# Visualization
# ----------------------------
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ----------------------------
# Progress / memory profiling
# ----------------------------
from tqdm import tqdm



# Building Multilayer Transition Matrix

In [ ]:
# Load all the matrices needed
OUTPUT_DIRECTORY = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"

## DGIDB
DGIDB_binary_matrix = sp.load_npz(DGIDB_DIRECTORY + "hypergraph_incidence_matrix_binary.npz")
DGIDB_weighted_matrix = sp.load_npz(DGIDB_DIRECTORY + "hypergraph_incidence_matrix_weighted.npz")
DGIDB_gene_weight_diag_matrix = sp.load_npz(DGIDB_DIRECTORY + "diag_gene_weight_matrix.npz")
DGIDB_diag_node_degree_matrix = sp.load_npz(DGIDB_DIRECTORY + "diag_node_degree_matrix.npz")
DGIDB_inverse_diag_edge_degree_matrix = sp.load_npz(
    DGIDB_DIRECTORY + "inverse_diag_edge_degree_matrix.npz"
    )

## MSIGDB
MSIGDB_binary_matrix = sp.load_npz(MSIGDB_DIRECTORY + "hypergraph_incidence_matrix_binary.npz")
MSIGDB_weighted_matrix = sp.load_npz(MSIGDB_DIRECTORY + "hypergraph_incidence_matrix_weighted.npz")
MSIGDB_gene_weight_diag_matrix = sp.load_npz(MSIGDB_DIRECTORY + "gene_weight_diag_matrix.npz")
MSIGDB_diag_node_degree_matrix = sp.load_npz(MSIGDB_DIRECTORY + "diag_node_degree_matrix.npz")
MSIGDB_inverse_diag_edge_degree_matrix = sp.load_npz(
    MSIGDB_DIRECTORY + "inverse_diag_edge_degree_matrix.npz"
    )

In [ ]:
# Useful Functions
def csr_equal_tol(A, B, atol=1e-8):
    # First check shapes and sparsity pattern
    if A.shape != B.shape or not np.array_equal(A.indptr, B.indptr) or not np.array_equal(A.indices, B.indices):
        return False
    # Compare numeric values within tolerance
    return np.allclose(A.data, B.data, atol=atol, rtol=0)


# row stochastic check
def row_stochastic_check(A):
    row_sums = np.array(A.sum(axis=1)).ravel()
    print(row_sums)
    ok = np.all(np.isclose(row_sums, 1.0))
    print("Every row sums to 1?", ok)
    return ok

def is_symmetric(W,tol = 1e-12):
    diff = (W - W.T)
    if len(np.abs(diff.data)) == 0:
        print("Matrix is exactly symmetric.")
        return True
    else:
        check = np.all(np.abs(diff.data) < tol)
        print(max(np.abs(diff.data)))
    return check

def degree_array(A, a=1):
    return np.asarray(A.sum(axis=a)).ravel()

def degree_diagonal_matrix(W, a=1):
    d = degree_array(W,a)
    return sp.diags(d, offsets=0, format='csr')

def symmetrically_normalize(W, a=1):
    D = np.asarray(W.sum(axis=a)).ravel()
    D_inv_sqrt = np.zeros_like(D)
    nze = D != 0
    D_inv_sqrt[nze] = 1 / np.sqrt(D[nze])

    W_sym = W.multiply(D_inv_sqrt)              # scale columns
    W_sym = W_sym.multiply(D_inv_sqrt[:, None]) # scale rows    
    return W_sym.tocsr()

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")
        
def relative_normality_error(A):
    if sp.issparse(A):
        AH = A.getH()
        AAH = A @ AH
        AHA = AH @ A
        C = AAH - AHA

        num = np.sqrt(np.sum(np.abs(C.data)**2))
        den = max(
            np.sqrt(np.sum(np.abs(AAH.data)**2)),
            np.sqrt(np.sum(np.abs(AHA.data)**2)),
            1e-30
        )
        return num / den
    else:
        AH = A.conj().T
        AAH = A @ AH
        AHA = AH @ A
        return np.linalg.norm(AAH - AHA, 'fro') / max(
            np.linalg.norm(AAH, 'fro'),
            np.linalg.norm(AHA, 'fro'),
            1e-30
        )

In [ ]:
# Building Adjacency Matrices

## DGIDB
H,W_v,D_v,D_e_inv, H_bin = DGIDB_weighted_matrix, DGIDB_gene_weight_diag_matrix, DGIDB_diag_node_degree_matrix, DGIDB_inverse_diag_edge_degree_matrix, DGIDB_binary_matrix

d = (D_v @ W_v).diagonal()
d_inv = np.zeros_like(d)
nonzero_mask = d > 0
d_inv[nonzero_mask] = 1.0 / d[nonzero_mask]
D_v_inv = sp.diags(d_inv)

DGIDB_adjacency_matrix = D_v_inv @ H @ D_e_inv @ H.T



In [ ]:
# ## MSIGDB
H,W_v,D_v,D_e_inv, H_bin = MSIGDB_weighted_matrix, MSIGDB_gene_weight_diag_matrix, MSIGDB_diag_node_degree_matrix, MSIGDB_inverse_diag_edge_degree_matrix, MSIGDB_binary_matrix

d = (D_v @ W_v).diagonal()
d_inv = np.zeros_like(d)
nonzero_mask = d > 0
d_inv[nonzero_mask] = 1.0 / d[nonzero_mask]
D_v_inv = sp.diags(d_inv)

MSIGDB_adjacency_matrix = D_v_inv @ H @ D_e_inv @ H.T

In [ ]:
assert(row_stochastic_check(DGIDB_adjacency_matrix))
assert(row_stochastic_check(MSIGDB_adjacency_matrix))

## coupling matrices

In [ ]:
## Build interlayer coupling matrices between the two layers

dgidb_to_msigdb_indices = []
# Open the JSON file and load its content into a dictionary
with open(DGIDB_DIRECTORY + "gene_to_index.json", "r") as file:
    dgidb = json.load(file)
with open(MSIGDB_DIRECTORY + "gene_to_index.json", "r") as file:
    msigdb = json.load(file)
DGIDB_index_to_gene = {index: gene for gene, index in dgidb.items()}
MSIGDB_index_to_gene = {index: gene for gene, index in msigdb.items()}

num_genes_msigdb = len(msigdb)
num_genes_dgidb = len(dgidb)

# Intialize the inter-layer matrix with zeros
C12 = np.zeros((num_genes_dgidb,num_genes_msigdb))
C21 = np.zeros((num_genes_msigdb,num_genes_dgidb))
B12_array = np.zeros(num_genes_dgidb)
B21_array = np.zeros(num_genes_msigdb)

i = 0
for gene_dgidb, idx_dgidb in dgidb.items():
    # If the gene exists in both gene-to-index mappings
    if gene_dgidb in msigdb:      
        idx_msigdb = msigdb[gene_dgidb]
        dgidb_to_msigdb_indices.append((idx_dgidb, idx_msigdb))
        i += 1
    else:
        dgidb_to_msigdb_indices.append((idx_dgidb, None))
        print(f"Gene {gene_dgidb} not found in MSIGDB mapping.")
        
# build C21 and C12

for idx_dgidb, idx_msigdb in dgidb_to_msigdb_indices:
    if (idx_msigdb is not None):
        C12[idx_dgidb,:] = MSIGDB_adjacency_matrix.getrow(idx_msigdb).toarray().ravel()
        C21[idx_msigdb,:] = DGIDB_adjacency_matrix.getrow(idx_dgidb).toarray().ravel()
        B12_array[idx_dgidb] = interlayer_transition_prob
        B21_array[idx_msigdb] = interlayer_transition_prob

A12_array = 1 - B12_array
A21_array = 1 - B21_array

B12 = sp.diags(B12_array)
B21 = sp.diags(B21_array)
A12 = sp.diags(A12_array)
A21 = sp.diags(A21_array)
# print stat
print(i/len(dgidb), "of DGIDB genes have a match in MSIGDB")
dgidb_to_msigdb_indices_dict = dict(dgidb_to_msigdb_indices) 

## Create gene to index ditinct

In [ ]:
# build DGIDB_to_real
n1,n2 = num_genes_dgidb,num_genes_msigdb
# dgidb_to_real = []
dgidb_idx = []
gene_to_index_dgidb_new= {}
coupling_matrix = C12
msigdb_idx = np.arange(n1,n1+n2)
num_additional_rows = 0

for didx, midx in dgidb_to_msigdb_indices_dict.items():
    if(midx is None):
        # dgidb_to_real.append(didx)
        dgidb_idx.append(didx) 
        gene_to_index_dgidb_new[DGIDB_index_to_gene[didx]] = num_additional_rows
        num_additional_rows += 1
    # else: 
    #     dgidb_to_real.append(n1 + midx)
    
MSIGDB_gene_to_index_new = {g: i + num_additional_rows for g, i in msigdb.items()}
gene_to_index_distinct = MSIGDB_gene_to_index_new | gene_to_index_dgidb_new   

In [ ]:
for u, v in dgidb_to_msigdb_indices_dict.items():
    print(1) if v is None else None

## construct MTM

In [ ]:
A = A12 @ DGIDB_adjacency_matrix
B = B12 @ C12
C = B21 @ C21
D = A21 @ MSIGDB_adjacency_matrix

In [ ]:
## Build the multilayer transition matrix
P = sp.bmat([
    [A, B],
    [C, D]
]).tocsr()

num_genes = P.shape[0]
# del A, B, C, D, DGIDB_adjacency_matrix, MSIGDB_adjacency_matrix
# for i in range(3):
#     gc.collect()

# Select columns and rows of P using dgidb idx list and msigdb idx list

In [ ]:
assert(row_stochastic_check(P))

## stationary distribution

In [ ]:
def stationary_distribution(W, tol=1e-9, maxit=20000, seed=0):
    # power iteration for left stationary of row-stochastic W
    n = W.shape[0]
    rng = np.random.default_rng(seed)
    pi = rng.random(n) + 1e-12
    pi /= pi.sum()
    for _ in range(maxit):
        pi_next = pi @ W
        if np.linalg.norm(pi_next - pi, 1) < tol:
            break
        pi = pi_next
    return pi / pi.sum()

In [ ]:
pi = stationary_distribution(P)

# Aggregation

## Column

In [ ]:
num_distinct_row = len(gene_to_index_distinct)
# total number of columns being aggregated from M
total_cols = n1 + n2

rows = np.empty(total_cols, dtype=int)
cols = np.empty(total_cols, dtype=int)
data = np.ones(total_cols, dtype=P.dtype)

# DGIDB part
for k in range(n1):
    rows[k] = k
    cols[k] = gene_to_index_distinct[DGIDB_index_to_gene[k]]

# MSIGDB part
for k in range(n2):
    rows[n1 + k] = n1 + k
    cols[n1 + k] = gene_to_index_distinct[MSIGDB_index_to_gene[k]]

# sparse grouping matrix
A_c = sp.csr_matrix((data, (rows, cols)), shape=(P.shape[1], num_distinct_row))

## Preparing Weights

In [ ]:
dgidb_pi_average = pi[0:n1].mean()
msigdb_pi_average = pi[n1:n1+n2].mean()
print("Average stationary probability for DGIDB genes:", dgidb_pi_average)
print("Average stationary probability for MSIGDB genes:", msigdb_pi_average)
print("Ratio of averages (DGIDB/MSIGDB):", dgidb_pi_average / msigdb_pi_average)

In [ ]:
pi

In [ ]:
wD_list,wM_list = [], []
for u, v in dgidb_to_msigdb_indices_dict.items():
    if (v is not None):
        pid = pi[u]
        pim = pi[dgidb_to_msigdb_indices_dict[u] + n1]
        wd = pid / (pid + pim)
        wm = pim / (pid + pim)
        wD_list.append(wd)
        wM_list.append(wm)

## Row

In [ ]:
wD_list2 = []
total_rows = n1 + n2

rows = np.empty(total_rows, dtype=int)
cols = np.empty(total_rows, dtype=int)
data = np.empty(total_rows, dtype=P.dtype)

# DGIDB part
for didx,midx in dgidb_to_msigdb_indices_dict.items():
    rows[didx] = didx
    cols[didx] = gene_to_index_distinct[DGIDB_index_to_gene[didx]]
    if (midx is None):
        data[didx] = 1.0
    else:
        wD = pi[didx] / (pi[didx] + pi[midx + n1])
        wD_list2.append(wD)
        data[didx] = wD

wM_list2_dict = {}
# MSIGDB part
for k in range(n2):
    rows[n1 + k] = n1 + k
    cols[n1 + k] = gene_to_index_distinct[MSIGDB_index_to_gene[k]]
    if (k in dgidb_to_msigdb_indices_dict.values()):
        # find the corresponding dgidb index
        corresponding_dgidb_idx = next(didx for didx, midx in dgidb_to_msigdb_indices_dict.items() if midx == k)
        wM = pi[n1 + k] / (pi[n1 + k] + pi[corresponding_dgidb_idx])
        wM_list2_dict[corresponding_dgidb_idx] = wM
        data[n1 + k] = wM
    else:
        data[n1 + k] = 1.0

# sparse grouping matrix
A_r = (sp.csr_matrix((data, (rows, cols)), shape=(P.shape[0], num_distinct_row))).T.tocsr()

In [ ]:
wM_list2 = [wM_list2_dict[k] for k in sorted(wM_list2_dict)]

In [ ]:
# Assert that the weights are computed the same as the graph
assert(wD_list == wD_list2 and wM_list == wM_list2)

# Gene Subsampling

In [ ]:
# idx of all dgidb genes in the aggregated matrix
dgidb_agg_idx_list = [gene_to_index_distinct[DGIDB_index_to_gene[idx]] for idx in range(n1)]

In [ ]:
# # randomly sample num_sample genes to be tested
# num_sample_dgidb = 100
# num_sample_msigdb = 500
# dgidb_idx_list = random.sample(range(DGIDB_adjacency_matrix.shape[0]), num_sample_dgidb)
# msigdb_idx_list = random.sample(range(MSIGDB_adjacency_matrix.shape[0]), num_sample_msigdb)
# print("Sampled DGIDB indices:", dgidb_idx_list)
# print("Sampled MSIGDB indices:", msigdb_idx_list)

In [ ]:
# # make indices into aggregated indices and combine them
# dgidb_agg_idx_list = [gene_to_index_distinct[DGIDB_index_to_gene[idx]] for idx in dgidb_idx_list]
# msigdb_agg_idx_list = [gene_to_index_distinct[MSIGDB_index_to_gene[idx]] for idx in msigdb_idx_list]
# combined_agg_idx_list = set(dgidb_agg_idx_list) | set(msigdb_agg_idx_list)
# print("Sampled DGIDB aggregated indices:", dgidb_agg_idx_list)
# print("Sampled MSIGDB aggregated indices:", msigdb_agg_idx_list)
# print("Combined aggregated indices:", combined_agg_idx_list)

In [ ]:
# # make sure combined lengths match
# print(len(combined_agg_idx_list))

In [ ]:
# # Save combined indices to a JSON file
# with open(OUTPUT_DIRECTORY + "/beta_testing/combined_agg_idx_list.json", "w") as file:
#     json.dump(list(combined_agg_idx_list), file)

# Avg distance to DGIDB genes for genes sampled

In [ ]:
# Load the combined indices from the JSON file
with open(OUTPUT_DIRECTORY + "/beta_testing/combined_agg_idx_list.json", "r") as file:
    combined_agg_idx_list = json.load(file)

In [ ]:
D_avg_beta = []

In [ ]:
# get selected rows of Ar P^t Ac for the sampled indices in a matrix free way through vector access
for idx in tqdm(combined_agg_idx_list):
    e = np.zeros(A_r.shape[0])
    e[idx] = 1.0

    idx_row = e @ A_r              # this is e_idx A_r
    avg_row = np.zeros(A_c.shape[1])

    current_t = 0

    for t in average_t:
        while current_t < t:
            idx_row = idx_row @ P  # reuse previous power
            current_t += 1
        avg_row += idx_row @ A_c   # this is e_idx A_r P^t A_c

    avg_row /= len(average_t)
    avg_dgidb_diff_dist = np.mean(avg_row[dgidb_agg_idx_list])
    # print(f"Average distance to DGIDB genes for aggregated index {idx}: {avg_dgidb_diff_dist}")
    D_avg_beta.append(avg_dgidb_diff_dist)

In [ ]:
# Convert the resulting list to nparray and save to a npy file
D_avg_beta_array = np.array(D_avg_beta)
np.save(OUTPUT_DIRECTORY + f"/beta_testing/D_avg_{interlayer_transition_prob}.npy", D_avg_beta_array)